# 🎙️ Voice Biometric Authentication System
### Complete Analysis Notebook — Preprocessing · Features · Verification · Spoof Detection
**B.Tech Semester 6 — Speech Processing Project**

Run each cell top-to-bottom. Works in Google Colab out of the box.

---
## ⚙️ Section 0 — Setup & Installation

In [ ]:
# Install all required packages
!pip install -q librosa soundfile noisereduce resemblyzer speechbrain matplotlib seaborn scikit-learn tqdm
print('All packages installed ✓')

In [ ]:
import os, sys, warnings, zipfile, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
from scipy.fft import dct as scipy_dct
from scipy.ndimage import uniform_filter1d
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#c9d1d9',
    'ytick.color':      '#c9d1d9',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'figure.titlesize': 14,
    'axes.titlesize':   12,
    'axes.titleweight': 'bold',
    'font.family':      'monospace',
})
ACCENT  = '#58a6ff'
GREEN   = '#3fb950'
RED     = '#f85149'
ORANGE  = '#d29922'
PURPLE  = '#bc8cff'
SR = 16000
print('Imports done ✓')

---
## 📂 Section 1 — Load Your Recordings
> **Option A** — Mount Google Drive (recommended)
> **Option B** — Upload a .zip of your recordings
> **Option C** — Auto-generates demo audio (no upload needed)

In [ ]:
# ── Option A: Mount Google Drive ────────────────────────────────
# Put your project1/data/ folder inside Google Drive and set the path below.
DRIVE_DATA_PATH = '/content/drive/MyDrive/voice_auth/data'   # ← change if needed

USE_DRIVE  = False   # set True if you've uploaded to Drive
USE_UPLOAD = False   # set True to upload a zip file
USE_DEMO   = True    # fallback: generates synthetic demo audio

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = DRIVE_DATA_PATH
    USE_DEMO = False

elif USE_UPLOAD:
    from google.colab import files
    print('Upload your data.zip file...')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content/')
    DATA_DIR = '/content/data'
    USE_DEMO = False

else:
    DATA_DIR = '/content/data'
    USE_DEMO = True

print(f'Data dir: {DATA_DIR}  |  Demo mode: {USE_DEMO}')

In [ ]:
# ── Generate demo audio if no real recordings ──────────────────
def make_real_voice(duration=4.0, fundamental=130.0, seed=0):
    """Voiced harmonic signal with natural jitter and syllabic modulation."""
    rng = np.random.default_rng(seed)
    t   = np.linspace(0, duration, int(SR * duration), endpoint=False)
    f0  = fundamental + 8 * np.sin(2*np.pi*0.4*t) + rng.normal(0, 0.8, len(t))
    phase = np.cumsum(2*np.pi*f0/SR)
    sig   = sum((1/k)*np.sin(k*phase) for k in range(1, 9))
    env   = 0.5 + 0.5*np.sin(2*np.pi*4.5*t)
    sig   = sig * env + 0.008*rng.standard_normal(len(t))
    sig  /= np.max(np.abs(sig))+1e-9
    return sig.astype(np.float32)

def make_fake_voice(seed=0):
    """TTS-like: constant F0, no modulation — anti-spoofing target."""
    t   = np.linspace(0, 4.0, SR*4, endpoint=False)
    sig = sum((1/k)*np.sin(2*np.pi*130*k*t) for k in range(1,8))
    sig /= np.max(np.abs(sig))+1e-9
    return sig.astype(np.float32)

if USE_DEMO:
    os.makedirs(f'{DATA_DIR}/raw/speaker/text_independent', exist_ok=True)
    os.makedirs(f'{DATA_DIR}/raw/impostor/text_independent', exist_ok=True)
    os.makedirs(f'{DATA_DIR}/spoof_data/real', exist_ok=True)
    os.makedirs(f'{DATA_DIR}/spoof_data/fake', exist_ok=True)
    for i in range(15):
        sf.write(f'{DATA_DIR}/raw/speaker/text_independent/sample_{i+1:02d}.wav',
                 make_real_voice(seed=i), SR)
        sf.write(f'{DATA_DIR}/spoof_data/real/sample_{i+1:02d}.wav',
                 make_real_voice(seed=i+100), SR)
        sf.write(f'{DATA_DIR}/spoof_data/fake/sample_{i+1:02d}.wav',
                 make_fake_voice(seed=i), SR)
    for i in range(5):
        sf.write(f'{DATA_DIR}/raw/impostor/text_independent/sample_{i+1:02d}.wav',
                 make_real_voice(fundamental=200, seed=i+50), SR)
    print('Demo audio generated: 15 real + 5 impostor + 15 real/fake spoof')

# Collect file lists
REAL_DIR = f'{DATA_DIR}/raw'
speaker_files = sorted(glob.glob(f'{REAL_DIR}/*/text_independent/*.wav') +
                       glob.glob(f'{REAL_DIR}/abhiram/text_independent/*.wav'))
speaker_files = sorted(set(speaker_files))
print(f'Found {len(speaker_files)} speaker recordings')

---
## 🔊 Section 2 — Preprocessing Pipeline
**Steps:** Raw audio → Noise Reduction → Silence Trimming → Normalization

In [ ]:
import noisereduce as nr

def preprocess(audio, sr=SR):
    audio = nr.reduce_noise(y=audio, sr=sr)
    audio, _ = librosa.effects.trim(audio, top_db=20)
    mx = np.max(np.abs(audio))
    if mx > 0: audio = audio / mx
    return audio.astype(np.float32)

# Load one sample for demonstration
demo_path = speaker_files[0]
raw_audio, _ = librosa.load(demo_path, sr=SR)
clean_audio   = preprocess(raw_audio)
print(f'Raw   : {len(raw_audio)/SR:.2f}s  max_amp={np.max(np.abs(raw_audio)):.4f}')
print(f'Clean : {len(clean_audio)/SR:.2f}s  max_amp={np.max(np.abs(clean_audio)):.4f}')

In [ ]:
# ── Preprocessing: Before vs After ─────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
fig.suptitle('Preprocessing Pipeline — Before vs After', fontsize=14, color='white', y=1.01)
t_raw   = np.linspace(0, len(raw_audio)/SR,   len(raw_audio))
t_clean = np.linspace(0, len(clean_audio)/SR, len(clean_audio))

# Row 1: Waveform
axes[0,0].plot(t_raw,   raw_audio,   color=ACCENT,  lw=0.5, alpha=0.8)
axes[0,0].set_title('Waveform — RAW');  axes[0,0].set_ylabel('Amplitude')
axes[0,1].plot(t_clean, clean_audio, color=GREEN,   lw=0.5, alpha=0.8)
axes[0,1].set_title('Waveform — CLEAN (noise reduced + trimmed + normalised)')

# Row 2: Spectrogram
for ax, audio, label, cmap in [
    (axes[1,0], raw_audio,   'Spectrogram — RAW',   'magma'),
    (axes[1,1], clean_audio, 'Spectrogram — CLEAN', 'viridis'),
]:
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
    img = librosa.display.specshow(D, sr=SR, hop_length=160, x_axis='time',
                                   y_axis='hz', ax=ax, cmap=cmap)
    ax.set_title(label); fig.colorbar(img, ax=ax, format='%+2.0f dB')

# Row 3: RMS Energy curve
for ax, audio, label, color in [
    (axes[2,0], raw_audio,   'RMS Energy — RAW',   ACCENT),
    (axes[2,1], clean_audio, 'RMS Energy — CLEAN', GREEN),
]:
    rms  = librosa.feature.rms(y=audio, hop_length=160)[0]
    t_r  = librosa.frames_to_time(np.arange(len(rms)), sr=SR, hop_length=160)
    ax.plot(t_r, rms, color=color, lw=1.5)
    ax.fill_between(t_r, rms, alpha=0.3, color=color)
    ax.set_title(label); ax.set_ylabel('RMS')

for ax in axes.flat:
    ax.set_xlabel('Time (s)'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('/content/01_preprocessing.png', dpi=120, bbox_inches='tight')
plt.show(); print('Saved: 01_preprocessing.png')

---
## 📊 Section 3 — Feature Extraction
Extracting: **MFCC · MFCC+Delta · Mel Spectrogram · Pitch F0 · Spectral · LFCC**

In [ ]:
# ── Feature extraction functions ────────────────────────────────
N_MFCC, N_MELS, N_LFCC = 40, 128, 60
HOP, WIN, NFFT = 160, 400, 512

def extract_mfcc(y):        return librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, hop_length=HOP)
def extract_mel(y):         return librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, hop_length=HOP), ref=np.max)
def extract_pitch(y):
    f0,_,_ = librosa.pyin(y, fmin=50, fmax=500, sr=SR, hop_length=HOP)
    return np.nan_to_num(f0, nan=0.0)
def extract_spectral(y):
    return {
        'centroid':  librosa.feature.spectral_centroid(y=y,  sr=SR, hop_length=HOP)[0],
        'bandwidth': librosa.feature.spectral_bandwidth(y=y, sr=SR, hop_length=HOP)[0],
        'rolloff':   librosa.feature.spectral_rolloff(y=y,   sr=SR, hop_length=HOP)[0],
        'zcr':       librosa.feature.zero_crossing_rate(y,          hop_length=HOP)[0],
    }
def _linear_fb(n_f, n_fft, sr):
    freq  = np.linspace(0, sr/2, n_fft//2+1)
    ctrs  = np.linspace(0, sr/2, n_f+2)
    fb = np.zeros((n_f, n_fft//2+1))
    for m in range(1, n_f+1):
        lo,mid,hi = ctrs[m-1],ctrs[m],ctrs[m+1]
        fb[m-1] = np.maximum(0, np.minimum((freq-lo)/(mid-lo+1e-9),(hi-freq)/(hi-mid+1e-9)))
    return fb
def extract_lfcc(y):
    fb  = _linear_fb(70, NFFT, SR)
    S   = np.abs(librosa.stft(y, n_fft=NFFT, hop_length=HOP, win_length=WIN))**2
    lf  = np.dot(fb, S)
    return scipy_dct(np.log(lf+1e-9), axis=0, norm='ortho')[:N_LFCC]

mfcc     = extract_mfcc(clean_audio)
mfcc_d   = librosa.feature.delta(mfcc)
mfcc_d2  = librosa.feature.delta(mfcc, order=2)
mel      = extract_mel(clean_audio)
pitch    = extract_pitch(clean_audio)
spectral = extract_spectral(clean_audio)
lfcc     = extract_lfcc(clean_audio)
t_frames = librosa.frames_to_time(np.arange(mfcc.shape[1]), sr=SR, hop_length=HOP)
print(f'MFCC shape       : {mfcc.shape}')
print(f'MFCC+delta shape : {np.vstack([mfcc,mfcc_d,mfcc_d2]).shape}')
print(f'Mel spec shape   : {mel.shape}')
print(f'Pitch shape      : {pitch.shape}')
print(f'LFCC shape       : {lfcc.shape}')

In [ ]:
# ── Feature Visualisation — All in one figure ───────────────────
fig = plt.figure(figsize=(18, 22))
fig.suptitle('Complete Feature Extraction', fontsize=16, color='white', y=1.005)
gs  = gridspec.GridSpec(6, 2, hspace=0.55, wspace=0.35)

# 1. Waveform
ax = fig.add_subplot(gs[0, :])
ax.plot(np.linspace(0,len(clean_audio)/SR,len(clean_audio)), clean_audio, color=ACCENT, lw=0.5)
ax.set_title('Preprocessed Waveform'); ax.set_ylabel('Amplitude'); ax.set_xlabel('Time (s)'); ax.grid(True, alpha=0.3)

# 2. Mel Spectrogram
ax = fig.add_subplot(gs[1, 0])
img = librosa.display.specshow(mel, sr=SR, hop_length=HOP, x_axis='time', y_axis='mel', ax=ax, cmap='magma')
ax.set_title('Mel Spectrogram (dB)'); fig.colorbar(img, ax=ax, format='%+2.0f dB')

# 3. MFCC
ax = fig.add_subplot(gs[1, 1])
img = librosa.display.specshow(mfcc, sr=SR, hop_length=HOP, x_axis='time', ax=ax, cmap='coolwarm')
ax.set_title(f'MFCCs ({N_MFCC} coefficients)'); ax.set_ylabel('MFCC #'); fig.colorbar(img, ax=ax)

# 4. MFCC Delta
ax = fig.add_subplot(gs[2, 0])
img = librosa.display.specshow(mfcc_d, sr=SR, hop_length=HOP, x_axis='time', ax=ax, cmap='RdBu_r')
ax.set_title('MFCC Δ (Delta)'); ax.set_ylabel('MFCC #'); fig.colorbar(img, ax=ax)

# 5. MFCC Delta-Delta
ax = fig.add_subplot(gs[2, 1])
img = librosa.display.specshow(mfcc_d2, sr=SR, hop_length=HOP, x_axis='time', ax=ax, cmap='RdBu_r')
ax.set_title('MFCC ΔΔ (Delta-Delta)'); ax.set_ylabel('MFCC #'); fig.colorbar(img, ax=ax)

# 6. LFCC
ax = fig.add_subplot(gs[3, 0])
img = librosa.display.specshow(lfcc, sr=SR, hop_length=HOP, x_axis='time', ax=ax, cmap='plasma')
ax.set_title(f'LFCC ({N_LFCC} coefficients — for anti-spoofing)'); ax.set_ylabel('LFCC #'); fig.colorbar(img, ax=ax)

# 7. Pitch F0
ax = fig.add_subplot(gs[3, 1])
voiced = pitch > 0
ax.plot(t_frames[voiced], pitch[voiced], 'o', color=PURPLE, markersize=1.5)
ax.fill_between(t_frames[voiced], pitch[voiced], alpha=0.2, color=PURPLE)
ax.set_title('Pitch Contour F0 (voiced frames only)'); ax.set_ylabel('Hz'); ax.set_xlabel('Time (s)'); ax.grid(True, alpha=0.3)

# 8. Spectral centroid & bandwidth
ax = fig.add_subplot(gs[4, 0])
ax.plot(t_frames[:len(spectral['centroid'])],  spectral['centroid'],  color=ACCENT,  lw=1.5, label='Centroid')
ax.plot(t_frames[:len(spectral['bandwidth'])], spectral['bandwidth'], color=ORANGE,  lw=1.5, label='Bandwidth')
ax.plot(t_frames[:len(spectral['rolloff'])],   spectral['rolloff'],   color=GREEN,   lw=1.5, label='Rolloff')
ax.set_title('Spectral Shape Features'); ax.set_ylabel('Hz'); ax.set_xlabel('Time (s)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 9. ZCR
ax = fig.add_subplot(gs[4, 1])
ax.plot(t_frames[:len(spectral['zcr'])], spectral['zcr'], color=RED, lw=1.5)
ax.fill_between(t_frames[:len(spectral['zcr'])], spectral['zcr'], alpha=0.2, color=RED)
ax.set_title('Zero Crossing Rate (ZCR)'); ax.set_ylabel('Rate'); ax.set_xlabel('Time (s)'); ax.grid(True, alpha=0.3)

# 10. Mean MFCC bar chart
ax = fig.add_subplot(gs[5, :])
means = mfcc.mean(axis=1); stds = mfcc.std(axis=1)
ax.bar(range(N_MFCC), means, color=ACCENT, alpha=0.8, label='Mean')
ax.errorbar(range(N_MFCC), means, yerr=stds, fmt='none', color='white', alpha=0.5, capsize=2, label='±Std')
ax.set_title('MFCC Mean ± Std (across time) — Speaker Fingerprint'); ax.set_xlabel('MFCC Coefficient #')
ax.set_ylabel('Value'); ax.legend(); ax.grid(True, alpha=0.3)

plt.savefig('/content/02_features.png', dpi=120, bbox_inches='tight')
plt.show(); print('Saved: 02_features.png')

---
## 🔍 Section 4 — Recording Quality Analysis
Checking all recordings: duration · RMS level · SNR · Voice Activity · Clipping

In [ ]:
# ── Analyse all recordings ──────────────────────────────────────
def analyse_file(path):
    y, _ = librosa.load(path, sr=SR)
    dur     = len(y)/SR
    rms_db  = 20*np.log10(np.sqrt(np.mean(y**2))+1e-9)
    peak    = float(np.max(np.abs(y)))
    frms    = librosa.feature.rms(y=y, hop_length=HOP)[0]
    fdb     = 20*np.log10(frms+1e-9)
    vad     = float((fdb > -40).mean())*100
    v_e     = frms[fdb>-40].mean() if (fdb>-40).any() else 1e-9
    s_e     = frms[fdb<=-40].mean() if (fdb<=-40).any() else 1e-9
    snr     = 20*np.log10(v_e/(s_e+1e-9))
    clp     = float((np.abs(y)>=0.999).mean())*100
    flags   = []
    if dur<1.5:     flags.append('TOO_SHORT')
    if rms_db<-35:  flags.append('TOO_QUIET')
    if vad<40:      flags.append('LOW_VAD')
    if snr<10:      flags.append('NOISY')
    if clp>1.0:     flags.append('CLIPPED')
    return dict(name=os.path.basename(path), dur=dur, rms=rms_db,
                peak=peak, vad=vad, snr=snr, clp=clp,
                ok=len(flags)==0, flags=flags)

results = [analyse_file(p) for p in speaker_files]
print(f'Analysed {len(results)} files')
print(f'\n{"File":<25} {"Dur":>5} {"RMS":>7} {"VAD%":>6} {"SNR":>7} {"Clip%":>6}  Status')
print('-'*70)
for r in results:
    st = ','.join(r['flags']) if r['flags'] else 'OK'
    print(f"{r['name']:<25} {r['dur']:>5.1f} {r['rms']:>7.1f} {r['vad']:>6.1f} {r['snr']:>7.1f} {r['clp']:>6.3f}  {st}")
ok_count = sum(1 for r in results if r['ok'])
print(f'\n  Total: {len(results)} | OK: {ok_count} | Issues: {len(results)-ok_count}')

In [ ]:
# ── Quality Dashboard ───────────────────────────────────────────
names = [r['name'].replace('.wav','') for r in results]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Recording Quality Dashboard', fontsize=14, color='white')

metrics = [
    ('dur',  'Duration (s)',         ACCENT,  (0,8),   3.0, 'min'),
    ('rms',  'RMS Level (dBFS)',      ORANGE, (-40,-10),-35, 'max'),
    ('vad',  'Voice Activity (%)',    GREEN,   (0,100),  40,  'min'),
    ('snr',  'SNR (dB)',              PURPLE,  (0,50),   10,  'min'),
    ('clp',  'Clipping (%)',          RED,     (0,2),    1.0, 'max'),
]
for ax, (key, label, color, ylim, thresh, direction) in zip(axes.flat, metrics):
    vals   = [r[key] for r in results]
    colors = [GREEN if r['ok'] else RED for r in results]
    bars   = ax.bar(range(len(results)), vals, color=colors, alpha=0.8, width=0.7)
    tline  = ax.axhline(thresh, color='yellow', lw=1.5, ls='--', label=f'Threshold={thresh}')
    ax.set_title(label); ax.set_ylim(ylim)
    ax.set_xticks(range(len(results))); ax.set_xticklabels(names, rotation=90, fontsize=6)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Overall pass/fail pie
ax = axes.flat[-1]
ok  = sum(1 for r in results if r['ok'])
bad = len(results) - ok
ax.pie([ok, bad], labels=[f'OK ({ok})', f'Issues ({bad})'],
       colors=[GREEN, RED], autopct='%1.0f%%', startangle=90,
       textprops={'color':'white','fontsize':12})
ax.set_title('Overall Quality')

plt.tight_layout(); plt.savefig('/content/03_quality_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Quality Heatmap ─────────────────────────────────────────────
import pandas as pd
df = pd.DataFrame(results)[['name','dur','rms','vad','snr','clp']]
df['name'] = df['name'].str.replace('.wav','')
df_norm = df.set_index('name')
# Normalise 0-1 (for heatmap display)
df_plot = df_norm.copy()
df_plot['dur'] = df_norm['dur']/8
df_plot['rms'] = (df_norm['rms']+40)/30     # -40 to -10 → 0 to 1
df_plot['vad'] = df_norm['vad']/100
df_plot['snr'] = df_norm['snr']/50
df_plot['clp'] = 1 - df_norm['clp']/2      # lower is better

fig, ax = plt.subplots(figsize=(14, max(6, len(results)*0.35)))
sns.heatmap(df_plot.T, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.3, linecolor='#0d1117',
            annot=df_norm.T.round(1), fmt='g', annot_kws={'size':7},
            cbar_kws={'label': 'Normalised Score (green=good)'})
ax.set_title('Recording Quality Heatmap (green = good, red = bad)', pad=12)
ax.set_xlabel('Recording'); ax.set_ylabel('Metric')
plt.tight_layout(); plt.savefig('/content/04_quality_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🎙️ Section 5 — Speaker Verification (Text-Independent)
Using **Resemblyzer GE2E** d-vector embeddings (256-dim). Enroll → Verify → Score.

In [ ]:
from resemblyzer import VoiceEncoder, preprocess_wav
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = VoiceEncoder(device=DEVICE)
print(f'Resemblyzer loaded on {DEVICE}')

In [ ]:
# ── Extract embeddings for all speaker recordings ───────────────
def get_embedding(path):
    y, _ = librosa.load(path, sr=SR)
    y = preprocess_wav(y, source_sr=SR)
    emb = encoder.embed_utterance(y)
    return emb / (np.linalg.norm(emb)+1e-9)

def cosine_sim(a, b): return float(np.dot(a/np.linalg.norm(a), b/np.linalg.norm(b)))

print('Extracting embeddings...')
embeddings = np.array([get_embedding(p) for p in speaker_files])
centroid   = embeddings.mean(axis=0)
centroid  /= np.linalg.norm(centroid)+1e-9
sims = [cosine_sim(e, centroid) for e in embeddings]
names_short = [os.path.basename(p).replace('.wav','') for p in speaker_files]

print(f'\n{"Recording":<25} {"Similarity":>10}  Verdict')
print('-'*50)
for name, sim in zip(names_short, sims):
    verdict = 'ACCEPTED ✓' if sim >= 0.75 else ('WEAK' if sim >= 0.55 else 'REJECTED ✗')
    print(f'{name:<25} {sim:>10.4f}  {verdict}')
print(f'\nMean similarity: {np.mean(sims):.4f}  Min: {np.min(sims):.4f}  Max: {np.max(sims):.4f}')

In [ ]:
# ── Embedding Similarity Plot ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Speaker Verification — Embedding Analysis', fontsize=14, color='white')

# Bar chart: similarity per sample
ax = axes[0]
bar_colors = [GREEN if s>=0.75 else (ORANGE if s>=0.55 else RED) for s in sims]
bars = ax.bar(range(len(sims)), sims, color=bar_colors, alpha=0.85, width=0.7)
ax.axhline(0.75, color='yellow', lw=2, ls='--', label='Threshold (0.75)')
ax.set_xticks(range(len(sims))); ax.set_xticklabels(names_short, rotation=90, fontsize=7)
ax.set_ylim(0, 1.05); ax.set_title('Cosine Similarity vs Enrolled Centroid')
ax.set_ylabel('Cosine Similarity'); ax.legend(); ax.grid(True, alpha=0.3)
# Annotate bars
for i, (bar, s) in enumerate(zip(bars, sims)):
    ax.text(bar.get_x()+bar.get_width()/2, s+0.01, f'{s:.2f}',
            ha='center', va='bottom', fontsize=6, color='white', rotation=90)

# 2D PCA of embeddings
from sklearn.decomposition import PCA
ax = axes[1]
pca  = PCA(n_components=2).fit_transform(embeddings)
sc   = ax.scatter(pca[:,0], pca[:,1], c=sims, cmap='RdYlGn',
                  vmin=0.5, vmax=1.0, s=80, edgecolors='white', lw=0.5)
ctrd = PCA(n_components=2).fit(embeddings).transform(centroid.reshape(1,-1))
ax.scatter(ctrd[:,0], ctrd[:,1], marker='*', s=300, color='yellow', zorder=5, label='Centroid')
for i, name in enumerate(names_short):
    ax.annotate(name, (pca[i,0], pca[i,1]), fontsize=6, color='#c9d1d9',
                xytext=(4,4), textcoords='offset points')
fig.colorbar(sc, ax=ax, label='Cosine Similarity')
ax.set_title('PCA of 256-dim Embeddings (2D)'); ax.legend()
ax.set_xlabel('PCA-1'); ax.set_ylabel('PCA-2'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('/content/05_speaker_verification.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Genuine vs Impostor Comparison ──────────────────────────────
# Load impostor files
impostor_files = sorted(glob.glob(f'{REAL_DIR}/impostor/text_independent/*.wav') +
                         glob.glob(f'{REAL_DIR}/friend/text_independent/*.wav') +
                         glob.glob(f'{REAL_DIR}/friend/*.wav'))
if not impostor_files:
    print('No impostor recordings found — generating synthetic impostors')
    impostor_files = [f'{DATA_DIR}/raw/impostor/text_independent/sample_{i+1:02d}.wav'
                      for i in range(5)]

imp_embs = np.array([get_embedding(p) for p in impostor_files])
imp_sims = [cosine_sim(e, centroid) for e in imp_embs]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Genuine vs Impostor Score Distribution', fontsize=14, color='white')

# Distribution plot
ax = axes[0]
ax.hist(sims, bins=15, color=GREEN,  alpha=0.7, label=f'Genuine (n={len(sims)})',  edgecolor='white')
ax.hist(imp_sims, bins=10, color=RED, alpha=0.7, label=f'Impostor (n={len(imp_sims)})', edgecolor='white')
ax.axvline(0.75, color='yellow', lw=2, ls='--', label='Threshold 0.75')
ax.set_title('Score Distribution'); ax.set_xlabel('Cosine Similarity'); ax.set_ylabel('Count')
ax.legend(); ax.grid(True, alpha=0.3)

# Scatter
ax = axes[1]
ax.scatter(range(len(sims)), sims, color=GREEN, s=60, label='Genuine', zorder=3)
ax.scatter(range(len(imp_sims)), imp_sims, color=RED, s=60, marker='x', lw=2, label='Impostor')
ax.axhline(0.75, color='yellow', lw=2, ls='--', label='Threshold')
ax.set_title('Per-Sample Scores'); ax.set_ylabel('Cosine Similarity')
ax.set_xlabel('Sample Index'); ax.legend(); ax.grid(True, alpha=0.3)

all_genuine  = sims
all_impostor = imp_sims
far = sum(s>=0.75 for s in all_impostor)/len(all_impostor)*100
frr = sum(s<0.75  for s in all_genuine) /len(all_genuine)*100
print(f'FAR (False Accept Rate): {far:.1f}%')
print(f'FRR (False Reject Rate): {frr:.1f}%')
print(f'Genuine mean:  {np.mean(all_genuine):.4f}')
print(f'Impostor mean: {np.mean(all_impostor):.4f}')
plt.tight_layout(); plt.savefig('/content/06_genuine_impostor.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🛡️ Section 6 — Anti-Spoofing / Deepfake Detection
Rule-based detector using LFCC + Pitch Jitter + Modulation Energy + Spectral Flatness

In [ ]:
# ── Spoof features on one real vs one fake ──────────────────────
real_path = sorted(glob.glob(f'{DATA_DIR}/spoof_data/real/*.wav'))[0]
fake_path = sorted(glob.glob(f'{DATA_DIR}/spoof_data/fake*.wav') +
                   glob.glob(f'{DATA_DIR}/spoof_data/fake_generated/*.wav'))[0]

real_audio, _ = librosa.load(real_path, sr=SR)
fake_audio, _ = librosa.load(fake_path, sr=SR)

def sub_band_flatness(y, bands=((0,500),(500,2000),(2000,4000),(4000,8000))):
    S = np.abs(librosa.stft(y, n_fft=NFFT, hop_length=HOP))**2
    freq = librosa.fft_frequencies(sr=SR, n_fft=NFFT)
    out = []
    for lo,hi in bands:
        mask = (freq>=lo)&(freq<hi)
        sub  = S[mask].mean(axis=1)
        geo  = np.exp(np.mean(np.log(sub+1e-9)))
        ari  = np.mean(sub)+1e-9
        out.append(geo/ari)
    return np.array(out)

def modulation_energy(y):
    env  = np.abs(librosa.effects.harmonic(y))
    hop2 = SR//100
    eds  = np.array([env[i:i+hop2].mean() for i in range(0,len(env)-hop2,hop2)])
    if len(eds)<32: return 0.0
    ms   = np.abs(np.fft.rfft(eds))
    mf   = np.fft.rfftfreq(len(eds), d=1/100)
    band = (mf>=4)&(mf<=16)
    return float(ms[band].sum()/(ms.sum()+1e-9))

def pitch_jitter(y):
    f0,_,_ = librosa.pyin(y, fmin=50, fmax=500, sr=SR, hop_length=HOP)
    f0 = np.nan_to_num(f0, nan=0.0)
    vf = f0[f0>0]
    if len(vf)<4: return 0.0
    return float(np.mean(np.abs(np.diff(vf)))/(np.mean(vf)+1e-9))

r_flat  = sub_band_flatness(real_audio)
f_flat  = sub_band_flatness(fake_audio)
r_mod   = modulation_energy(real_audio)
f_mod   = modulation_energy(fake_audio)
r_jit   = pitch_jitter(real_audio)
f_jit   = pitch_jitter(fake_audio)
print(f'{'Feature':<25} {'Real':>10} {'Fake':>10}  Higher=Real?')
print('-'*58)
print(f"{'Pitch Jitter':<25} {r_jit:>10.4f} {f_jit:>10.4f}  {r_jit>f_jit}")
print(f"{'Modulation Energy':<25} {r_mod:>10.4f} {f_mod:>10.4f}  {r_mod>f_mod}")
for i,(lo,hi) in enumerate([(0,500),(500,2000),(2000,4000),(4000,8000)]):
    print(f"  Flatness {lo}-{hi}Hz       {r_flat[i]:>10.4f} {f_flat[i]:>10.4f}  {r_flat[i]<f_flat[i]} (lower=natural)")

In [ ]:
# ── Real vs Fake Feature Comparison Plot ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Real vs Fake (TTS/Vocoder) Feature Comparison', fontsize=14, color='white')

# 1. Waveforms
for ax, audio, label, color in [
    (axes[0,0], real_audio, 'Waveform — REAL', GREEN),
    (axes[0,1], fake_audio, 'Waveform — FAKE (vocoder)', RED),
]:
    t = np.linspace(0, len(audio)/SR, len(audio))
    ax.plot(t, audio, color=color, lw=0.5)
    ax.set_title(label); ax.set_ylabel('Amplitude'); ax.set_xlabel('Time (s)'); ax.grid(True,alpha=0.3)

# 2. Spectrograms
for ax, audio, label, cmap in [
    (axes[1,0], real_audio, 'Spectrogram — REAL', 'Greens'),
    (axes[1,1], fake_audio, 'Spectrogram — FAKE', 'Reds'),
]:
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio, n_fft=NFFT, hop_length=HOP)), ref=np.max)
    librosa.display.specshow(D, sr=SR, hop_length=HOP, x_axis='time', y_axis='hz', ax=ax, cmap=cmap)
    ax.set_title(label)

# 3. Feature bar comparison
ax = axes[0,2]
feature_names = ['Jitter (×100)', 'Mod Energy', 'Flat 0-500', 'Flat 0.5-2k', 'Flat 2-4k', 'Flat 4-8k']
real_vals = [r_jit*100, r_mod] + list(r_flat)
fake_vals = [f_jit*100, f_mod] + list(f_flat)
x = np.arange(len(feature_names)); w=0.35
ax.bar(x-w/2, real_vals, w, label='Real', color=GREEN, alpha=0.8)
ax.bar(x+w/2, fake_vals, w, label='Fake', color=RED,   alpha=0.8)
ax.set_title('Anti-Spoofing Features — Real vs Fake')
ax.set_xticks(x); ax.set_xticklabels(feature_names, rotation=30, ha='right', fontsize=8)
ax.legend(); ax.grid(True, alpha=0.3)

# 4. LFCC comparison
ax = axes[1,2]
r_lfcc = extract_lfcc(real_audio); f_lfcc = extract_lfcc(fake_audio)
ax.plot(r_lfcc.mean(axis=1), color=GREEN, lw=2, label='Real mean LFCC')
ax.fill_between(range(N_LFCC), r_lfcc.mean(axis=1)-r_lfcc.std(axis=1),
                 r_lfcc.mean(axis=1)+r_lfcc.std(axis=1), alpha=0.2, color=GREEN)
ax.plot(f_lfcc.mean(axis=1), color=RED, lw=2, label='Fake mean LFCC')
ax.fill_between(range(N_LFCC), f_lfcc.mean(axis=1)-f_lfcc.std(axis=1),
                 f_lfcc.mean(axis=1)+f_lfcc.std(axis=1), alpha=0.2, color=RED)
ax.set_title('LFCC Profile — Real vs Fake'); ax.set_xlabel('LFCC #'); ax.set_ylabel('Value')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('/content/07_spoof_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Rule-Based Spoof Scores on All Files ─────────────────────────
def rule_based_score(y):
    jit  = pitch_jitter(y)
    mod  = modulation_energy(y)
    flat = sub_band_flatness(y)
    harmonic = librosa.effects.harmonic(y, margin=3.0)
    hnr  = float(np.mean(harmonic**2)/(np.mean(y**2)+1e-9))
    S    = np.abs(librosa.stft(y,n_fft=NFFT,hop_length=HOP))**2
    freq = librosa.fft_frequencies(sr=SR, n_fft=NFFT)
    tot  = S.sum()+1e-9
    hi   = S[freq>=4000].sum()/tot
    jit_s  = float(np.clip(jit/0.02,0,1))
    mod_s  = float(np.clip(mod/0.25,0,1))
    flat_s = float(np.clip(1-flat[2:].mean()*2,0,1))
    hnr_s  = float(np.clip(1-abs(hnr-0.70)/0.30,0,1))
    hi_s   = float(np.clip(hi/0.15,0,1))
    return 0.30*jit_s + 0.25*mod_s + 0.20*flat_s + 0.15*hnr_s + 0.10*hi_s

real_files = sorted(glob.glob(f'{DATA_DIR}/spoof_data/real/*.wav'))
fake_files = sorted(glob.glob(f'{DATA_DIR}/spoof_data/fake_generated/*.wav') or
                    glob.glob(f'{DATA_DIR}/spoof_data/fake/*.wav'))

print('Computing spoof scores...')
real_scores = []; fake_scores = []
for p in real_files[:15]:
    y,_ = librosa.load(p, sr=SR)
    real_scores.append(rule_based_score(y))
for p in fake_files[:15]:
    y,_ = librosa.load(p, sr=SR)
    fake_scores.append(rule_based_score(y))

thresh = (np.mean(real_scores)+np.mean(fake_scores))/2
print(f'Real scores — mean={np.mean(real_scores):.4f}  std={np.std(real_scores):.4f}')
print(f'Fake scores — mean={np.mean(fake_scores):.4f}  std={np.std(fake_scores):.4f}')
print(f'Midpoint threshold: {thresh:.4f}')
far_s = sum(s>=thresh for s in fake_scores)/len(fake_scores)*100
frr_s = sum(s<thresh  for s in real_scores)/len(real_scores)*100
print(f'Spoof FAR={far_s:.1f}%  FRR={frr_s:.1f}%')

In [ ]:
# ── Spoof Score Distribution Plot ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Anti-Spoofing Score Analysis', fontsize=14, color='white')

ax = axes[0]
ax.hist(real_scores, bins=12, color=GREEN, alpha=0.75, label=f'Real (n={len(real_scores)})', edgecolor='white')
ax.hist(fake_scores, bins=12, color=RED,   alpha=0.75, label=f'Fake (n={len(fake_scores)})', edgecolor='white')
ax.axvline(thresh, color='yellow', lw=2, ls='--', label=f'Threshold={thresh:.3f}')
ax.set_title('Score Distribution — Real vs Fake')
ax.set_xlabel('Anti-Spoof Score (higher=more real)')
ax.set_ylabel('Count'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(range(len(real_scores)), real_scores, color=GREEN, s=70, label='Real', zorder=3)
ax.scatter(range(len(fake_scores)), fake_scores, color=RED,   s=70, marker='x', lw=2, label='Fake')
ax.axhline(thresh, color='yellow', lw=2, ls='--', label=f'Threshold={thresh:.3f}')
ax.set_title('Per-Sample Spoof Scores')
ax.set_xlabel('Sample'); ax.set_ylabel('Score'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('/content/08_spoof_scores.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🤖 Section 7 — GMM Spoof Detector (Train + Evaluate)

In [ ]:
# ── Extract fixed-length feature vectors for GMM ─────────────────
def get_spoof_vector(y):
    lfcc = extract_lfcc(y)             # (60,T)
    mf   = extract_mfcc(y)             # (40,T)
    md   = librosa.feature.delta(mf)
    sp   = extract_spectral(y)
    flat = sub_band_flatness(y)
    jit  = pitch_jitter(y)
    mod  = modulation_energy(y)
    vec  = np.concatenate([
        lfcc.mean(axis=1), lfcc.std(axis=1),
        flat, [jit, mod],
        [sp['centroid'].mean(), sp['centroid'].std()],
        [sp['zcr'].mean(), sp['zcr'].std()],
        md.std(axis=1)[:8],
    ])
    return vec.astype(np.float32)

print('Extracting GMM features...')
X_real = np.vstack([get_spoof_vector(librosa.load(p,sr=SR)[0]) for p in real_files[:15]])
X_fake = np.vstack([get_spoof_vector(librosa.load(p,sr=SR)[0]) for p in fake_files[:15]])
print(f'Real feature matrix: {X_real.shape}')
print(f'Fake feature matrix: {X_fake.shape}')

In [ ]:
# ── Train GMM ────────────────────────────────────────────────────
n_comp = max(1, min(8, len(X_real)//2, len(X_fake)//2))
scaler = StandardScaler().fit(np.vstack([X_real, X_fake]))
Xr_s   = scaler.transform(X_real).astype(np.float64)
Xf_s   = scaler.transform(X_fake).astype(np.float64)

gmm_real = GaussianMixture(n_components=n_comp, covariance_type='diag',
                            max_iter=200, reg_covar=1e-3, random_state=42).fit(Xr_s)
gmm_fake = GaussianMixture(n_components=n_comp, covariance_type='diag',
                            max_iter=200, reg_covar=1e-3, random_state=42).fit(Xf_s)
print(f'GMM trained: {n_comp} components per class')

def gmm_score(y):
    v  = scaler.transform(get_spoof_vector(y).reshape(1,-1)).astype(np.float64)
    ll = gmm_real.score(v) - gmm_fake.score(v)
    return float(1/(1+np.exp(-ll/10)))

gmm_real_s = [gmm_score(librosa.load(p,sr=SR)[0]) for p in real_files[:15]]
gmm_fake_s = [gmm_score(librosa.load(p,sr=SR)[0]) for p in fake_files[:15]]
g_thresh   = (np.mean(gmm_real_s)+np.mean(gmm_fake_s))/2
print(f'GMM Real scores — mean={np.mean(gmm_real_s):.4f}')
print(f'GMM Fake scores — mean={np.mean(gmm_fake_s):.4f}')
far_g = sum(s>=g_thresh for s in gmm_fake_s)/len(gmm_fake_s)*100
frr_g = sum(s<g_thresh  for s in gmm_real_s)/len(gmm_real_s)*100
print(f'GMM FAR={far_g:.1f}%  FRR={frr_g:.1f}%')

In [ ]:
# ── GMM Scores Plot ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('GMM Spoof Detector Results', fontsize=14, color='white')

ax = axes[0]
ax.hist(gmm_real_s, bins=10, color=GREEN, alpha=0.75, label=f'Real (n={len(gmm_real_s)})', edgecolor='white')
ax.hist(gmm_fake_s, bins=10, color=RED,   alpha=0.75, label=f'Fake (n={len(gmm_fake_s)})', edgecolor='white')
ax.axvline(g_thresh, color='yellow', lw=2, ls='--', label=f'Threshold={g_thresh:.3f}')
ax.set_title('GMM Score Distribution'); ax.set_xlabel('Score'); ax.legend(); ax.grid(True,alpha=0.3)

ax = axes[1]
all_s  = gmm_real_s + gmm_fake_s
labels = ['Real']*len(gmm_real_s) + ['Fake']*len(gmm_fake_s)
colors_s = [GREEN]*len(gmm_real_s) + [RED]*len(gmm_fake_s)
ax.scatter(range(len(all_s)), all_s, c=colors_s, s=60, edgecolors='white', lw=0.5)
ax.axhline(g_thresh, color='yellow', lw=2, ls='--')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=GREEN,label='Real'),Patch(color=RED,label='Fake'),
                   plt.Line2D([],[],color='yellow',ls='--',label=f'Thresh={g_thresh:.2f}')])
ax.set_title('GMM Scores per Sample'); ax.set_xlabel('Sample'); ax.grid(True,alpha=0.3)

# Confusion matrix
ax = axes[2]
preds_r = [1 if s>=g_thresh else 0 for s in gmm_real_s]
preds_f = [1 if s>=g_thresh else 0 for s in gmm_fake_s]
tp=sum(preds_r); fn=len(preds_r)-tp
fp=sum(preds_f); tn=len(preds_f)-fp
cm_arr = np.array([[tp,fn],[fp,tn]])
sns.heatmap(cm_arr, annot=True, fmt='d', cmap='RdYlGn', ax=ax,
            xticklabels=['Pred Real','Pred Fake'],
            yticklabels=['True Real','True Fake'],
            cbar=False, linewidths=2, linecolor='#0d1117',
            annot_kws={'size':16, 'color':'white', 'weight':'bold'})
acc = (tp+tn)/(tp+tn+fp+fn)*100
ax.set_title(f'Confusion Matrix  (Acc={acc:.1f}%)')

plt.tight_layout(); plt.savefig('/content/09_gmm_results.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🏁 Section 8 — Full Authentication Pipeline & Summary

In [ ]:
# ── End-to-End Authentication on each recording ──────────────────
SPOOF_THRESH = thresh      # from rule-based
VOICE_THRESH = 0.75        # speaker verification

print(f'\n{"Recording":<28} {"Spoof":>8} {"Voice":>8}  Decision')
print('='*62)
pipeline_results = []
for path in speaker_files:
    y, _ = librosa.load(path, sr=SR)
    y_c  = y / (np.max(np.abs(y))+1e-9)
    sp   = rule_based_score(y_c)
    emb  = get_embedding(path)
    vs   = cosine_sim(emb, centroid)
    if sp < SPOOF_THRESH:
        decision = 'REJECTED_SPOOF'
    elif vs < VOICE_THRESH:
        decision = 'REJECTED_VOICE'
    else:
        decision = 'ACCEPTED'
    pipeline_results.append((os.path.basename(path), sp, vs, decision))
    icon = '✓' if decision=='ACCEPTED' else '✗'
    print(f'{os.path.basename(path):<28} {sp:>8.4f} {vs:>8.4f}  {icon} {decision}')

accepted = sum(1 for _,_,_,d in pipeline_results if d=='ACCEPTED')
rej_s    = sum(1 for _,_,_,d in pipeline_results if d=='REJECTED_SPOOF')
rej_v    = sum(1 for _,_,_,d in pipeline_results if d=='REJECTED_VOICE')
print(f'\nAccepted: {accepted}  Spoof-rejected: {rej_s}  Voice-rejected: {rej_v}')

In [ ]:
# ── Final Summary Dashboard ──────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Voice Biometric Authentication — Final Results Dashboard', fontsize=16, color='white', y=1.01)
gs = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

# 1. Pipeline outcome per sample
ax = fig.add_subplot(gs[0, :])
names_p  = [r[0].replace('.wav','') for r in pipeline_results]
spoof_sc = [r[1] for r in pipeline_results]
voice_sc = [r[2] for r in pipeline_results]
dec_col  = [GREEN if r[3]=='ACCEPTED' else (ORANGE if r[3]=='REJECTED_VOICE' else RED)
            for r in pipeline_results]
x = np.arange(len(names_p)); w=0.35
ax.bar(x-w/2, spoof_sc, w, color=[c+'88' for c in dec_col], label='Spoof score', alpha=0.9)
ax.bar(x+w/2, voice_sc, w, color=dec_col, alpha=0.9, label='Voice score')
ax.axhline(SPOOF_THRESH, color=ORANGE, lw=1.5, ls=':', label=f'Spoof thresh={SPOOF_THRESH:.2f}')
ax.axhline(VOICE_THRESH, color='yellow', lw=1.5, ls='--', label=f'Voice thresh={VOICE_THRESH:.2f}')
ax.set_xticks(x); ax.set_xticklabels(names_p, rotation=90, fontsize=7)
ax.set_title('Full Pipeline Scores per Recording (green bar = ACCEPTED)'); ax.legend(fontsize=8)
ax.set_ylabel('Score [0-1]'); ax.set_ylim(0,1.05); ax.grid(True, alpha=0.3)

# 2. Decision pie
ax = fig.add_subplot(gs[1, 0])
vals  = [accepted, rej_v, rej_s]
clrs  = [GREEN, ORANGE, RED]
lbls  = [f'ACCEPTED ({accepted})', f'Rejected-Voice ({rej_v})', f'Rejected-Spoof ({rej_s})']
ax.pie(vals, labels=lbls, colors=clrs, autopct='%1.0f%%', startangle=90,
       textprops={'color':'white', 'fontsize':10})
ax.set_title('Decision Breakdown')

# 3. Score scatter: spoof vs voice
ax = fig.add_subplot(gs[1, 1])
ax.scatter(spoof_sc, voice_sc, c=dec_col, s=80, edgecolors='white', lw=0.5, zorder=3)
ax.axvline(SPOOF_THRESH, color=ORANGE, lw=1.5, ls=':')
ax.axhline(VOICE_THRESH, color='yellow', lw=1.5, ls='--')
ax.set_xlabel('Anti-Spoof Score'); ax.set_ylabel('Voice Score')
ax.set_title('Spoof vs Voice Score Space'); ax.grid(True, alpha=0.3)
# Label accept zone
ax.fill_between([SPOOF_THRESH,1],[VOICE_THRESH,VOICE_THRESH],[1,1], alpha=0.07, color=GREEN)
ax.text(0.88, 0.88, 'ACCEPT\nZONE', ha='center', va='center', color=GREEN, fontsize=9, transform=ax.transAxes)

# 4. Metrics summary table
ax = fig.add_subplot(gs[1, 2])
ax.axis('off')
metrics_tbl = [
    ['Metric', 'Value'],
    ['Total recordings', str(len(results))],
    ['Quality OK', f'{sum(1 for r in results if r["ok"])}/{len(results)}'],
    ['Enroll samples', str(len(speaker_files))],
    ['Mean embed sim', f'{np.mean(sims):.4f}'],
    ['Min embed sim',  f'{np.min(sims):.4f}'],
    ['Voice FAR', f'{far:.1f}%'],
    ['Voice FRR', f'{frr:.1f}%'],
    ['Spoof FAR',  f'{far_s:.1f}%'],
    ['Spoof FRR',  f'{frr_s:.1f}%'],
    ['GMM Accuracy', f'{acc:.1f}%'],
    ['Pipeline Accept', f'{accepted}/{len(pipeline_results)}'],
]
tbl = ax.table(cellText=metrics_tbl[1:], colLabels=metrics_tbl[0],
               cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for (r,c), cell in tbl.get_celld().items():
    cell.set_facecolor('#161b22' if r>0 else '#21262d')
    cell.set_text_props(color='white'); cell.set_edgecolor('#30363d')
ax.set_title('System Metrics Summary')

plt.savefig('/content/10_final_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 📈 Section 9 — Accuracy & Evaluation Metrics
EER · ROC Curve · Precision · Recall · F1 · Confusion Matrix · Full Summary Table

In [ ]:

# ── Section 9: Accuracy & Evaluation Metrics ─────────────────────
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                              average_precision_score, classification_report,
                              confusion_matrix)

# ─────────────────────────────────────────────────────────────────
# Helper: Equal Error Rate
# ─────────────────────────────────────────────────────────────────
def compute_eer(genuine_scores, impostor_scores):
    """Compute EER from genuine and impostor score lists."""
    y_true  = [1]*len(genuine_scores) + [0]*len(impostor_scores)
    y_score = list(genuine_scores)   + list(impostor_scores)
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.abs(fnr - fpr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2) * 100
    eer_thresh = float(thresholds[eer_idx])
    return eer, eer_thresh, fpr, tpr, thresholds, auc(fpr, tpr)


# ─────────────────────────────────────────────────────────────────
# 1. SPEAKER VERIFICATION METRICS
# ─────────────────────────────────────────────────────────────────
genuine_scores  = sims          # cosine sims of YOUR voice (from Section 5)
impostor_scores = imp_sims      # cosine sims of impostors

eer_v, eer_thresh_v, fpr_v, tpr_v, _, auc_v = compute_eer(genuine_scores, impostor_scores)

# at fixed threshold 0.75
sv_tp = sum(s >= 0.75 for s in genuine_scores)
sv_fn = sum(s <  0.75 for s in genuine_scores)
sv_fp = sum(s >= 0.75 for s in impostor_scores)
sv_tn = sum(s <  0.75 for s in impostor_scores)

sv_acc   = (sv_tp + sv_tn) / (sv_tp+sv_fn+sv_fp+sv_tn) * 100
sv_prec  = sv_tp / (sv_tp + sv_fp + 1e-9) * 100
sv_rec   = sv_tp / (sv_tp + sv_fn + 1e-9) * 100
sv_f1    = 2*sv_prec*sv_rec / (sv_prec+sv_rec+1e-9)
sv_far   = sv_fp / (sv_fp + sv_tn + 1e-9) * 100
sv_frr   = sv_fn / (sv_fn + sv_tp + 1e-9) * 100

print('=' * 55)
print('  SPEAKER VERIFICATION  (Resemblyzer GE2E, thresh=0.75)')
print('=' * 55)
print(f'  Accuracy          : {sv_acc:.1f}%')
print(f'  Precision         : {sv_prec:.1f}%')
print(f'  Recall            : {sv_rec:.1f}%')
print(f'  F1 Score          : {sv_f1:.1f}%')
print(f'  FAR               : {sv_far:.1f}%')
print(f'  FRR               : {sv_frr:.1f}%')
print(f'  EER               : {eer_v:.1f}%  (at thresh={eer_thresh_v:.3f})')
print(f'  AUC-ROC           : {auc_v:.4f}')
print(f'  Genuine mean sim  : {np.mean(genuine_scores):.4f}')
print(f'  Impostor mean sim : {np.mean(impostor_scores):.4f}')
print(f'  Score gap         : {np.mean(genuine_scores)-np.mean(impostor_scores):.4f}')


# ─────────────────────────────────────────────────────────────────
# 2. ANTI-SPOOFING METRICS  (Rule-Based)
# ─────────────────────────────────────────────────────────────────
eer_s, eer_thresh_s, fpr_s, tpr_s, _, auc_s = compute_eer(real_scores, fake_scores)

sp_tp = sum(s >= thresh for s in real_scores)
sp_fn = sum(s <  thresh for s in real_scores)
sp_fp = sum(s >= thresh for s in fake_scores)
sp_tn = sum(s <  thresh for s in fake_scores)

sp_acc  = (sp_tp + sp_tn) / (sp_tp+sp_fn+sp_fp+sp_tn) * 100
sp_prec = sp_tp / (sp_tp + sp_fp + 1e-9) * 100
sp_rec  = sp_tp / (sp_tp + sp_fn + 1e-9) * 100
sp_f1   = 2*sp_prec*sp_rec / (sp_prec+sp_rec+1e-9)
sp_far  = sp_fp / (sp_fp + sp_tn + 1e-9) * 100
sp_frr  = sp_fn / (sp_fn + sp_tp + 1e-9) * 100

print()
print('=' * 55)
print('  ANTI-SPOOFING  (Rule-Based: Jitter + Modulation + LFCC)')
print('=' * 55)
print(f'  Accuracy          : {sp_acc:.1f}%')
print(f'  Precision         : {sp_prec:.1f}%')
print(f'  Recall            : {sp_rec:.1f}%')
print(f'  F1 Score          : {sp_f1:.1f}%')
print(f'  FAR               : {sp_far:.1f}%')
print(f'  FRR               : {sp_frr:.1f}%')
print(f'  EER               : {eer_s:.1f}%  (at thresh={eer_thresh_s:.3f})')
print(f'  AUC-ROC           : {auc_s:.4f}')
print(f'  Real score mean   : {np.mean(real_scores):.4f}  std={np.std(real_scores):.4f}')
print(f'  Fake score mean   : {np.mean(fake_scores):.4f}  std={np.std(fake_scores):.4f}')


# ─────────────────────────────────────────────────────────────────
# 3. GMM ANTI-SPOOFING METRICS
# ─────────────────────────────────────────────────────────────────
eer_g, eer_thresh_g, fpr_g, tpr_g, _, auc_g = compute_eer(gmm_real_s, gmm_fake_s)

gm_tp = sum(s >= g_thresh for s in gmm_real_s)
gm_fn = sum(s <  g_thresh for s in gmm_real_s)
gm_fp = sum(s >= g_thresh for s in gmm_fake_s)
gm_tn = sum(s <  g_thresh for s in gmm_fake_s)

gm_acc  = (gm_tp + gm_tn) / (gm_tp+gm_fn+gm_fp+gm_tn) * 100
gm_prec = gm_tp / (gm_tp + gm_fp + 1e-9) * 100
gm_rec  = gm_tp / (gm_tp + gm_fn + 1e-9) * 100
gm_f1   = 2*gm_prec*gm_rec / (gm_prec+gm_rec+1e-9)
gm_far  = gm_fp / (gm_fp + gm_tn + 1e-9) * 100
gm_frr  = gm_fn / (gm_fn + gm_tp + 1e-9) * 100

print()
print('=' * 55)
print('  ANTI-SPOOFING  (GMM Classifier)')
print('=' * 55)
print(f'  Accuracy          : {gm_acc:.1f}%')
print(f'  Precision         : {gm_prec:.1f}%')
print(f'  Recall            : {gm_rec:.1f}%')
print(f'  F1 Score          : {gm_f1:.1f}%')
print(f'  FAR               : {gm_far:.1f}%')
print(f'  FRR               : {gm_frr:.1f}%')
print(f'  EER               : {eer_g:.1f}%  (at thresh={eer_thresh_g:.3f})')
print(f'  AUC-ROC           : {auc_g:.4f}')


# ─────────────────────────────────────────────────────────────────
# 4. FULL PIPELINE METRICS
# ─────────────────────────────────────────────────────────────────
# All pipeline_results are genuine voice samples (your voice)
# correct decision = ACCEPTED
pipe_correct = sum(1 for _,_,_,d in pipeline_results if d == 'ACCEPTED')
pipe_acc     = pipe_correct / len(pipeline_results) * 100

print()
print('=' * 55)
print('  FULL PIPELINE  (Spoof + Voice combined)')
print('=' * 55)
print(f'  Total samples     : {len(pipeline_results)}')
print(f'  Correctly accepted: {pipe_correct}')
print(f'  Spoof-rejected    : {rej_s}  (false negatives — your voice flagged as fake)')
print(f'  Voice-rejected    : {rej_v}  (wrong speaker)')
print(f'  Pipeline Accuracy : {pipe_acc:.1f}%')
print(f'  Spoof threshold   : {SPOOF_THRESH:.4f}')
print(f'  Voice threshold   : {VOICE_THRESH:.2f}')


In [ ]:

# ── Accuracy Visualisations: ROC + Confusion Matrices + Summary ──
fig = plt.figure(figsize=(20, 16))
fig.suptitle('Complete Evaluation Metrics', fontsize=16, color='white', y=1.01)
gs  = gridspec.GridSpec(3, 3, hspace=0.50, wspace=0.38)

# ── 1. ROC curves (all 3 systems) ────────────────────────────────
ax = fig.add_subplot(gs[0, :2])
ax.plot(fpr_v, tpr_v, color=GREEN,  lw=2.5, label=f'Speaker Verif  AUC={auc_v:.3f}  EER={eer_v:.1f}%')
ax.plot(fpr_s, tpr_s, color=ORANGE, lw=2.5, label=f'Rule-Based Spoof  AUC={auc_s:.3f}  EER={eer_s:.1f}%')
ax.plot(fpr_g, tpr_g, color=PURPLE, lw=2.5, label=f'GMM Spoof  AUC={auc_g:.3f}  EER={eer_g:.1f}%')
ax.plot([0,1],[0,1],'--', color='#555', lw=1, label='Random (AUC=0.50)')
ax.scatter([sv_far/100],[sv_rec/100], color=GREEN,  s=120, zorder=5, marker='D')
ax.scatter([sp_far/100],[sp_rec/100], color=ORANGE, s=120, zorder=5, marker='D')
ax.scatter([gm_far/100],[gm_rec/100], color=PURPLE, s=120, zorder=5, marker='D')
ax.set_title('ROC Curves — All Subsystems  (◆ = operating point at chosen threshold)')
ax.set_xlabel('False Positive Rate (FAR)'); ax.set_ylabel('True Positive Rate (1-FRR)')
ax.set_xlim(0,1); ax.set_ylim(0,1.02); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── 2. EER comparison bar ─────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
systems = ['Speaker\nVerif', 'Rule-Based\nSpoof', 'GMM\nSpoof']
eers    = [eer_v, eer_s, eer_g]
aucs    = [auc_v, auc_s, auc_g]
bar_c   = [GREEN, ORANGE, PURPLE]
bars = ax.bar(systems, eers, color=bar_c, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, eers):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.3, f'{val:.1f}%',
            ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
ax.set_title('EER Comparison (lower = better)'); ax.set_ylabel('EER (%)'); ax.grid(True, alpha=0.3)
ax2 = ax.twinx()
ax2.plot(systems, aucs, 'o--', color='yellow', lw=2, markersize=8, label='AUC-ROC')
ax2.set_ylim(0,1.1); ax2.set_ylabel('AUC-ROC', color='yellow')
ax2.tick_params(axis='y', colors='yellow')
ax2.legend(loc='lower right')

# ── 3. Confusion Matrix — Speaker Verification ────────────────────
ax = fig.add_subplot(gs[1, 0])
cm_sv = np.array([[sv_tp, sv_fn],[sv_fp, sv_tn]])
sns.heatmap(cm_sv, annot=True, fmt='d', cmap='Greens', ax=ax, cbar=False,
            linewidths=2, linecolor='#0d1117',
            xticklabels=['Pred Accept','Pred Reject'],
            yticklabels=['True Genuine','True Impostor'],
            annot_kws={'size':16,'color':'white','weight':'bold'})
ax.set_title(f'Speaker Verification\nAcc={sv_acc:.1f}%  F1={sv_f1:.1f}%')

# ── 4. Confusion Matrix — Rule-Based Spoof ────────────────────────
ax = fig.add_subplot(gs[1, 1])
cm_sp = np.array([[sp_tp, sp_fn],[sp_fp, sp_tn]])
sns.heatmap(cm_sp, annot=True, fmt='d', cmap='Oranges', ax=ax, cbar=False,
            linewidths=2, linecolor='#0d1117',
            xticklabels=['Pred Real','Pred Fake'],
            yticklabels=['True Real','True Fake'],
            annot_kws={'size':16,'color':'white','weight':'bold'})
ax.set_title(f'Rule-Based Spoof\nAcc={sp_acc:.1f}%  F1={sp_f1:.1f}%')

# ── 5. Confusion Matrix — GMM ─────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
cm_gm = np.array([[gm_tp, gm_fn],[gm_fp, gm_tn]])
sns.heatmap(cm_gm, annot=True, fmt='d', cmap='Purples', ax=ax, cbar=False,
            linewidths=2, linecolor='#0d1117',
            xticklabels=['Pred Real','Pred Fake'],
            yticklabels=['True Real','True Fake'],
            annot_kws={'size':16,'color':'white','weight':'bold'})
ax.set_title(f'GMM Spoof\nAcc={gm_acc:.1f}%  F1={gm_f1:.1f}%')

# ── 6. Score distributions with EER line ─────────────────────────
for ax_idx, (g_sc, i_sc, eer_val, eer_t, label, color1, color2) in enumerate([
    (genuine_scores, impostor_scores, eer_v, eer_thresh_v, 'Speaker Verification', GREEN, RED),
    (real_scores,    fake_scores,     eer_s, eer_thresh_s,  'Rule-Based Spoof',    GREEN, RED),
    (gmm_real_s,     gmm_fake_s,      eer_g, eer_thresh_g,  'GMM Spoof',           GREEN, RED),
]):
    ax = fig.add_subplot(gs[2, ax_idx])
    ax.hist(g_sc, bins=10, color=color1, alpha=0.65, label='Genuine/Real', edgecolor='white')
    ax.hist(i_sc, bins=10, color=color2, alpha=0.65, label='Impostor/Fake', edgecolor='white')
    ax.axvline(eer_t, color='yellow', lw=2, ls='-.',  label=f'EER thresh={eer_t:.3f}')
    ax.set_title(f'{label}\nScore Distribution'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.set_xlabel('Score'); ax.set_ylabel('Count')

plt.savefig('/content/11_accuracy_metrics.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: 11_accuracy_metrics.png')


# ── 7. Final summary table (printed) ─────────────────────────────
print()
print('╔' + '═'*65 + '╗')
print('║{:^65}║'.format('COMPLETE EVALUATION SUMMARY'))
print('╠' + '═'*65 + '╣')
rows = [
    ('Component',          'Acc%',  'F1%',  'EER%', 'AUC',   'FAR%', 'FRR%'),
    ('─'*22,               '─'*5,   '─'*5,  '─'*5,  '─'*5,   '─'*5,  '─'*5),
    ('Speaker Verif',      f'{sv_acc:.1f}', f'{sv_f1:.1f}', f'{eer_v:.1f}', f'{auc_v:.3f}', f'{sv_far:.1f}', f'{sv_frr:.1f}'),
    ('Rule-Based Spoof',   f'{sp_acc:.1f}', f'{sp_f1:.1f}', f'{eer_s:.1f}', f'{auc_s:.3f}', f'{sp_far:.1f}', f'{sp_frr:.1f}'),
    ('GMM Spoof',          f'{gm_acc:.1f}', f'{gm_f1:.1f}', f'{eer_g:.1f}', f'{auc_g:.3f}', f'{gm_far:.1f}', f'{gm_frr:.1f}'),
    ('─'*22,               '─'*5,   '─'*5,  '─'*5,  '─'*5,   '─'*5,  '─'*5),
    ('Full Pipeline',      f'{pipe_acc:.1f}', '—', '—', '—', '—', '—'),
]
for row in rows:
    print('║  {:<22}  {:>5}  {:>5}  {:>5}  {:>5}  {:>5}  {:>5}  ║'.format(*row))
print('╚' + '═'*65 + '╝')
print(f'\n  Genuine–Impostor gap : {np.mean(genuine_scores)-np.mean(impostor_scores):.4f}  (speaker verif)')
print(f'  Real–Fake score gap  : {np.mean(real_scores)-np.mean(fake_scores):.4f}  (rule-based spoof)')


---
## 🔐 Section 10 — Text-Dependent Authentication
**Passphrase:** *"My voice is my password"*

Three-stage pipeline: **Anti-spoofing → Passphrase check (Whisper ASR) → Speaker verification**

In [ ]:

# ── Install Whisper ──────────────────────────────────────────────
!pip install -q openai-whisper
print('Whisper installed ✓')


In [ ]:

# ── Load Whisper model ───────────────────────────────────────────
import whisper, re
from difflib import SequenceMatcher

PASSPHRASE      = "my voice is my password"
TEXT_THRESHOLD  = 0.70    # fuzzy match score to accept passphrase
VOICE_THRESHOLD = 0.75    # cosine similarity to accept speaker

whisper_model = whisper.load_model("base")
print(f'Whisper "base" model loaded ✓')
print(f'Passphrase : "{PASSPHRASE}"')


# ── Text similarity helpers ──────────────────────────────────────
def _norm(t):
    t = t.lower()
    t = re.sub(r"[^a-z0-9\s]", "", t)
    return " ".join(t.split())

def word_overlap(a, b):
    wa, wb = set(a.split()), set(b.split())
    return len(wa & wb) / len(wb) if wb else 0.0

def text_similarity(transcript, expected):
    t, e = _norm(transcript), _norm(expected)
    seq  = SequenceMatcher(None, t, e).ratio()
    wov  = word_overlap(t, e)
    return 0.6 * seq + 0.4 * wov

def transcribe(audio):
    a = audio.astype(np.float32)
    r = whisper_model.transcribe(a, language="en", fp16=False, verbose=False)
    return r["text"].strip()


In [ ]:

# ── Load passphrase recordings ───────────────────────────────────
td_files = sorted(
    glob.glob(f'{DATA_DIR}/raw/abhiram/text_dependent/*.wav') +
    glob.glob(f'{DATA_DIR}/raw/abhiram/text_dep/*.wav') +
    glob.glob(f'{DATA_DIR}/raw/*/text_dependent/*.wav') +
    glob.glob(f'{DATA_DIR}/enroll_0[1-3].wav') +
    glob.glob(f'{DATA_DIR}/enroll_07.wav') +
    glob.glob(f'{DATA_DIR}/enroll_08.wav') +
    glob.glob(f'{DATA_DIR}/enroll_09.wav')
)
td_files = sorted(set(td_files))

# If no real passphrase files found, generate demo ones
if not td_files:
    print('No passphrase recordings found — generating demo audio...')
    demo_td_dir = f'{DATA_DIR}/raw/demo/text_dependent'
    os.makedirs(demo_td_dir, exist_ok=True)
    for i in range(5):
        # Simulate 'my voice is my password' — voiced with varying F0
        t_arr = np.linspace(0, 4.0, SR*4, endpoint=False)
        f0    = 130 + 5*np.sin(2*np.pi*0.5*t_arr)
        phase = np.cumsum(2*np.pi*f0/SR)
        sig   = sum((1/k)*np.sin(k*phase) for k in range(1, 8))
        env   = 0.5 + 0.5*np.sin(2*np.pi*3.5*t_arr)
        sig   = (sig * env).astype(np.float32)
        sig  /= np.max(np.abs(sig)) + 1e-9
        path  = f'{demo_td_dir}/passphrase_{i+1:02d}.wav'
        sf.write(path, sig, SR)
        td_files.append(path)
    print(f'  Generated {len(td_files)} demo passphrase files')

print(f'Found {len(td_files)} text-dependent recording(s):')
for f in td_files:
    print(f'  {os.path.basename(f)}')


In [ ]:

# ── Step 1: Transcribe all passphrase recordings ─────────────────
print(f'Transcribing {len(td_files)} passphrase recordings with Whisper...\n')
print(f'{"File":<20} {"Transcript":<45} {"Text Score":>10}  Verdict')
print('-' * 85)

td_audios      = []
td_transcripts = []
td_text_scores = []
td_text_pass   = []

for path in td_files:
    y, _  = librosa.load(path, sr=SR)
    y_c   = preprocess(y)
    td_audios.append(y_c)

    transcript  = transcribe(y_c)
    score       = text_similarity(transcript, PASSPHRASE)
    accepted    = score >= TEXT_THRESHOLD

    td_transcripts.append(transcript)
    td_text_scores.append(score)
    td_text_pass.append(accepted)

    verdict = '✓ PASS' if accepted else '✗ FAIL'
    name    = os.path.basename(path)
    print(f'{name:<20} {transcript:<45} {score:>10.4f}  {verdict}')

print(f'\nText check:  {sum(td_text_pass)}/{len(td_text_pass)} passed  '
      f'(threshold={TEXT_THRESHOLD})')


In [ ]:

# ── Step 2: Build speaker centroid from passphrase recordings ────
print('Building speaker centroid from passphrase recordings...')
td_embeddings = np.array([get_embedding(p) for p in td_files])
td_centroid   = td_embeddings.mean(axis=0)
td_centroid  /= np.linalg.norm(td_centroid) + 1e-9
print(f'Centroid shape: {td_centroid.shape}')

# Speaker scores for each passphrase recording vs centroid
td_voice_scores = [cosine_sim(e, td_centroid) for e in td_embeddings]


# ── Step 3: Full text-dependent decision per file ────────────────
print(f'\n{"File":<20} {"Text":>8} {"Voice":>8}  Decision')
print('=' * 58)

td_results = []
for i, path in enumerate(td_files):
    ts   = td_text_scores[i]
    vs   = td_voice_scores[i]
    sp   = rule_based_score(td_audios[i])

    if sp < 0.20:
        decision = 'REJECTED_SPOOF'
    elif ts < TEXT_THRESHOLD:
        decision = 'REJECTED_TEXT'
    elif vs < VOICE_THRESHOLD:
        decision = 'REJECTED_VOICE'
    else:
        decision = 'ACCEPTED'

    td_results.append((os.path.basename(path), ts, vs, sp, decision))
    icon = '✓' if decision == 'ACCEPTED' else '✗'
    print(f'{os.path.basename(path):<20} {ts:>8.4f} {vs:>8.4f}  {icon} {decision}')

n_acc   = sum(1 for *_, d in td_results if d == 'ACCEPTED')
n_t_rej = sum(1 for *_, d in td_results if d == 'REJECTED_TEXT')
n_v_rej = sum(1 for *_, d in td_results if d == 'REJECTED_VOICE')
n_s_rej = sum(1 for *_, d in td_results if d == 'REJECTED_SPOOF')
print(f'\nAccepted: {n_acc}  Text-rejected: {n_t_rej}  '
      f'Voice-rejected: {n_v_rej}  Spoof-rejected: {n_s_rej}')


In [ ]:

# ── Text-Dependent Analysis Dashboard ───────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.suptitle('Text-Dependent Authentication Analysis', fontsize=16, color='white', y=1.01)
gs  = gridspec.GridSpec(2, 3, hspace=0.50, wspace=0.38)

names_td   = [r[0].replace('.wav','') for r in td_results]
ts_vals    = [r[1] for r in td_results]
vs_vals    = [r[2] for r in td_results]
sp_vals    = [r[3] for r in td_results]
decisions  = [r[4] for r in td_results]
dec_colors = [GREEN if d=='ACCEPTED' else (ORANGE if d=='REJECTED_TEXT' else RED)
              for d in decisions]

# 1. Text scores per file
ax = fig.add_subplot(gs[0, 0])
bars = ax.bar(range(len(names_td)), ts_vals, color=dec_colors, alpha=0.85, edgecolor='white')
ax.axhline(TEXT_THRESHOLD, color='yellow', lw=2, ls='--', label=f'Text thresh={TEXT_THRESHOLD}')
ax.set_xticks(range(len(names_td)))
ax.set_xticklabels(names_td, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1.1); ax.set_title('Passphrase Text Score per Recording')
ax.set_ylabel('Fuzzy Match Score'); ax.legend(); ax.grid(True, alpha=0.3)

# 2. Voice scores per file
ax = fig.add_subplot(gs[0, 1])
bars = ax.bar(range(len(names_td)), vs_vals, color=dec_colors, alpha=0.85, edgecolor='white')
ax.axhline(VOICE_THRESHOLD, color='yellow', lw=2, ls='--', label=f'Voice thresh={VOICE_THRESHOLD}')
ax.set_xticks(range(len(names_td)))
ax.set_xticklabels(names_td, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1.1); ax.set_title('Speaker Voice Score (Resemblyzer)')
ax.set_ylabel('Cosine Similarity'); ax.legend(); ax.grid(True, alpha=0.3)

# 3. Combined score (text 40% + voice 60%)
ax = fig.add_subplot(gs[0, 2])
combined = [0.4*t + 0.6*v for t, v in zip(ts_vals, vs_vals)]
bars = ax.bar(range(len(names_td)), combined, color=dec_colors, alpha=0.85, edgecolor='white')
ax.axhline(0.4*TEXT_THRESHOLD + 0.6*VOICE_THRESHOLD, color='yellow', lw=2, ls='--',
           label='Combined thresh')
ax.set_xticks(range(len(names_td)))
ax.set_xticklabels(names_td, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1.1); ax.set_title('Combined Score  (40%×text + 60%×voice)')
ax.set_ylabel('Score'); ax.legend(); ax.grid(True, alpha=0.3)

# 4. Transcription quality — text score scatter
ax = fig.add_subplot(gs[1, 0])
ax.scatter(range(len(names_td)), ts_vals, c=dec_colors, s=120, edgecolors='white', lw=0.8, zorder=3)
ax.axhline(TEXT_THRESHOLD, color='yellow', lw=2, ls='--', label=f'Threshold {TEXT_THRESHOLD}')
for i, (name, ts) in enumerate(zip(names_td, ts_vals)):
    ax.annotate(f'{ts:.2f}', (i, ts), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8, color='white')
ax.set_xticks(range(len(names_td)))
ax.set_xticklabels(names_td, rotation=45, ha='right', fontsize=8)
ax.set_title('Passphrase Match Score (per file)'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)

# 5. Text vs Voice scatter (decision space)
ax = fig.add_subplot(gs[1, 1])
ax.scatter(ts_vals, vs_vals, c=dec_colors, s=120, edgecolors='white', lw=0.8, zorder=3)
ax.axvline(TEXT_THRESHOLD,  color=ORANGE, lw=1.5, ls=':', label=f'Text thresh={TEXT_THRESHOLD}')
ax.axhline(VOICE_THRESHOLD, color='yellow', lw=1.5, ls='--', label=f'Voice thresh={VOICE_THRESHOLD}')
for i, name in enumerate(names_td):
    ax.annotate(name, (ts_vals[i], vs_vals[i]), fontsize=7, color='#c9d1d9',
                xytext=(4, 4), textcoords='offset points')
ax.fill_between([TEXT_THRESHOLD, 1], [VOICE_THRESHOLD, VOICE_THRESHOLD], [1, 1],
                alpha=0.07, color=GREEN)
ax.text(0.85, 0.9, 'ACCEPT\nZONE', ha='center', color=GREEN,
        fontsize=9, transform=ax.transAxes)
ax.set_xlabel('Text Score'); ax.set_ylabel('Voice Score')
ax.set_title('Text vs Voice Score Space'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 6. Pipeline decision summary
ax = fig.add_subplot(gs[1, 2])
from matplotlib.patches import Patch
labels_pie  = [f'ACCEPTED ({n_acc})',
               f'Text Fail ({n_t_rej})',
               f'Voice Fail ({n_v_rej})',
               f'Spoof Fail ({n_s_rej})']
sizes_pie   = [n_acc, n_t_rej, n_v_rej, n_s_rej]
colors_pie  = [GREEN, ORANGE, RED, '#ff9500']
non_zero    = [(l, s, c) for l, s, c in zip(labels_pie, sizes_pie, colors_pie) if s > 0]
if non_zero:
    lbl, sz, clr = zip(*non_zero)
    ax.pie(sz, labels=lbl, colors=clr, autopct='%1.0f%%', startangle=90,
           textprops={'color': 'white', 'fontsize': 10})
ax.set_title('Text-Dependent Pipeline\nDecision Breakdown')

plt.savefig('/content/12_text_dependent.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: 12_text_dependent.png')


# ── Comparison: Text-Independent vs Text-Dependent ───────────────
print()
print('╔' + '═'*55 + '╗')
print('║{:^55}║'.format('TEXT-INDEPENDENT  vs  TEXT-DEPENDENT'))
print('╠' + '═'*55 + '╣')
rows2 = [
    ('Mode',                  'Samples', 'Accepted', 'Accuracy'),
    ('─'*20,                  '─'*7,     '─'*8,      '─'*8),
    ('Text-Independent',      str(len(pipeline_results)),
                              str(accepted),
                              f'{accepted/len(pipeline_results)*100:.1f}%'),
    ('Text-Dependent',        str(len(td_results)),
                              str(n_acc),
                              f'{n_acc/len(td_results)*100:.1f}%' if td_results else '—'),
]
for row in rows2:
    print('║  {:<20}  {:>7}  {:>8}  {:>8}  ║'.format(*row))
print('╚' + '═'*55 + '╝')


In [ ]:
# ── Save all plots as a zip ──────────────────────────────────────
import zipfile, glob as g2
plots = g2.glob('/content/*.png')
with zipfile.ZipFile('/content/voice_auth_plots.zip','w') as zf:
    for p in sorted(plots):
        zf.write(p, os.path.basename(p))
print(f'Saved {len(plots)} plots to /content/voice_auth_plots.zip')
print('Download it from the Colab file browser (left panel → Files).')

---
## ✅ Notebook Complete

| Plot file | Contents |
|---|---|
| `01_preprocessing.png` | Waveform + Spectrogram + RMS before/after |
| `02_features.png` | MFCC · MFCC-delta · Mel Spec · Pitch · LFCC · Spectral |
| `03_quality_dashboard.png` | Per-file quality bar charts + pass/fail pie |
| `04_quality_heatmap.png` | Heatmap of all metrics across all recordings |
| `05_speaker_verification.png` | Embedding similarity + PCA scatter |
| `06_genuine_impostor.png` | Score distribution + FAR/FRR |
| `07_spoof_comparison.png` | Real vs fake waveform + spectrogram + features |
| `08_spoof_scores.png` | Rule-based scores distribution |
| `09_gmm_results.png` | GMM scores + confusion matrix |
| `10_final_dashboard.png` | Full pipeline decision + metrics summary |